# Notebook 06: Unique Advanced Analyses

This notebook saves reproducible figure exports and a summary CSV. Each analysis is guarded so the notebook can run even if the cleaned data is not present.

In [45]:
# Imports and setup
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

output_dir = Path('../reports/figures')
output_dir.mkdir(parents=True, exist_ok=True)

# Robust loader: try several likely locations so notebook runs from different cwd's
candidates = [Path('data/weather_cleaned.csv'), Path('../data/weather_cleaned.csv'), Path('notebooks/../data/weather_cleaned.csv')]
data_path = None
for p in candidates:
    if p.exists():
        data_path = p
        break
# fallback: any CSV under ./data or ../data
if data_path is None:
    for base in [Path('data'), Path('../data')]:
        if base.exists():
            csvs = list(base.glob('*.csv'))
            if csvs:
                data_path = csvs[0]
                break
if data_path is None:
    print('No cleaned CSV found under common data paths — analyses will be skipped. Place weather_cleaned.csv in a data/ folder.')
    df = pd.DataFrame()
else:
    df = pd.read_csv(data_path, parse_dates=['last_updated'])
    print(f"Loaded dataset: {df.shape} from {data_path}")
    if 'last_updated' in df.columns:
        print(f"Date range: {df['last_updated'].min()} to {df['last_updated'].max()}")

Loaded dataset: (143457, 59) from ../data/weather_cleaned.csv
Date range: 2024-05-16 01:45:00 to 2026-05-25 19:30:00


## Analysis 1 — Climate Pattern by Country

In [46]:
if df.empty:
    print('Skipping climate pattern analysis — no data')
else:
    df['year'] = df['last_updated'].dt.year
    climate_trends = df.groupby(['year','country'])['temperature_celsius'].mean().reset_index()
    top_countries = df['country'].value_counts().head(5).index.tolist()
    sample = climate_trends[climate_trends['country'].isin(top_countries)]
    fig = px.line(sample, x='year', y='temperature_celsius', color='country', markers=True, title='Avg Temp Trends (Top Countries)')
    fig.write_html(output_dir / '06_climate_trends.html')
    print('Saved: 06_climate_trends.html')

Saved: 06_climate_trends.html


## Analysis 2 — AQI vs Weather

In [47]:
if df.empty:
    print('Skipping AQI analysis — no data')
else:
    aqi_col = 'air_quality_us-epa-index'
    if aqi_col in df.columns:
        env = df[['temperature_celsius','humidity',aqi_col]].dropna()
        corr = env.corr(numeric_only=True)
        fig = px.imshow(corr, text_auto=True, title='Weather vs AQI Correlation')
        fig.write_html(output_dir / '06_aqi_correlation.html')
        print('Saved: 06_aqi_correlation.html')
    else:
        print('AQI column not present — skipping')

Saved: 06_aqi_correlation.html


## Analysis 3 — Tree-based Feature Importance (proxy)

In [48]:
if df.empty:
    print('Skipping feature importance — no data')
else:
    city_col = 'location_name' if 'location_name' in df.columns else 'country'
    city = df[city_col].dropna().mode().iat[0]
    city_data = df[df[city_col]==city].set_index('last_updated')
    ts = city_data['temperature_celsius'].resample('D').mean().interpolate()
    lag_df = pd.DataFrame({'y':ts})
    for lag in [1,2,3,7,14]:
        lag_df[f'lag_{lag}'] = ts.shift(lag)
    lag_df['rolling7'] = ts.shift(1).rolling(7).mean()
    lag_df = lag_df.dropna()
    if len(lag_df)<10:
        print('Not enough data for feature importance')
    else:
        X = lag_df.drop('y',axis=1)
        y = lag_df['y']
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X,y)
        fi = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
        fig = px.bar(fi.head(10).reset_index(), x='index', y=0, labels={'index':'feature','0':'importance'}, title=f'Feature Importance ({city})')
        fig.write_html(output_dir / '06_feature_importance.html')
        print('Saved: 06_feature_importance.html')

Saved: 06_feature_importance.html


## Analysis 4 — Spatial Heatmap (city-level)

In [49]:
if df.empty:
    print('Skipping spatial heatmap — no data')
else:
    ct = df.groupby('location_name').agg({'temperature_celsius':'mean','latitude':'first','longitude':'first'}).reset_index().dropna()
    if ct.empty:
        print('No city coords available')
    else:
        mn = ct['temperature_celsius'].min()
        ct['marker'] = (ct['temperature_celsius'] - mn).clip(lower=0) + 1
        fig = px.scatter_geo(ct, lat='latitude', lon='longitude', size='marker', color='temperature_celsius', hover_name='location_name', title='City Temperature Distribution', size_max=18)
        fig.write_html(output_dir / '06_spatial_heatmap.html')
        print('Saved: 06_spatial_heatmap.html')

Saved: 06_spatial_heatmap.html


## Summary & Export

In [50]:
if df.empty:
    print('No data — nothing exported')
else:
    print('Unique Analyses Summary')
    print(f" Total locations: {df['location_name'].nunique()}")
    print(f" Total countries: {df['country'].nunique()}")
    # prefer writing to ./data, fallback to ../data; create if missing
    out_dir = None
    for cand in [Path('data'), Path('../data')]:
        if cand.exists():
            out_dir = cand
            break
    if out_dir is None:
        out_dir = Path('data')
        out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / 'weather_unique_analyses.csv'
    df.to_csv(out_file, index=False)
    print(f'Saved: {out_file}')

Unique Analyses Summary
 Total locations: 257
 Total countries: 211
Saved: ../data/weather_unique_analyses.csv
